# Chapitre 2 · Les nombres qui apprennent

Notebook du chapitre 2 de *Construire un LLM de zéro*. Au chapitre 1, tu as
entraîné MiniLM : 91 497 nombres qui écrivent du pseudo-La Fontaine. Ici, tu
ouvres son fichier et tu apprends à le lire : vecteurs, produit scalaire,
produit matriciel.

**Comment travailler.** La leçon d'abord : tout le code du chapitre, complet et
prêt à exécuter, notion par notion. Lis, exécute, modifie pour voir. À la fin,
la section **Exercices** : trois défis à trous, du plus simple au plus costaud,
validés par des `assert`.

Tout tourne **sans GPU et sans connexion internet**, en quelques secondes.

In [1]:
import torch

_ = torch.manual_seed(42)                # mêmes nombres aléatoires à chaque exécution

## 1. Ouvre le fichier

Un modèle, c'est un fichier de nombres (les **weights**) et une notice
(l'architecture). La cellule suivante recrée un MiniLM identique à celui du
chapitre 1 : mêmes cinq paquets, mêmes shapes. Ses poids sont aléatoires (les
bonnes valeurs, elles, viennent de l'entraînement), mais pour lire le fichier,
seules les formes comptent.

`torch.save` écrit le dictionnaire des poids sur le disque, `torch.load` le
relit : c'est très exactement ce que tu télécharges quand tu « télécharges un
modèle ».

In [2]:
import torch.nn as nn

vocab_size = 81                          # les trente fables du chapitre 1 : 81 caractères distincts
block_size = 16                          # le modèle lit 16 caractères pour prédire le 17e


class MiniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, 24)          # chaque caractère -> 24 nombres
        self.reseau = nn.Sequential(
            nn.Linear(block_size * 24, 192),               # mélange tout le contexte
            nn.Tanh(),                                     # petite non-linéarité
            nn.Linear(192, vocab_size),                    # un score par caractère possible
        )

    def forward(self, x):
        return self.reseau(self.table(x).flatten(1))


model = MiniLM()
torch.save(model.state_dict(), "minilm.pt")    # à la fin du chapitre 1

etat = torch.load("minilm.pt")
total = 0
for nom, t in etat.items():
    print(f"{nom:16s}  shape {str(tuple(t.shape)):12s}  {t.numel():6d} nombres")
    total += t.numel()

print(f"{'':16s}  {'':18s}  ------")
print(f"{'total':16s}  {'':18s}  {total:6d} nombres")
assert total == 91_497, "le fichier de MiniLM doit contenir 91 497 nombres"
assert len(etat) == 5, "cinq paquets : trois matrices et deux vecteurs"

table.weight      shape (81, 24)        1944 nombres
reseau.0.weight   shape (192, 384)     73728 nombres
reseau.0.bias     shape (192,)           192 nombres
reseau.2.weight   shape (81, 192)      15552 nombres
reseau.2.bias     shape (81,)             81 nombres
                                      ------
total                                  91497 nombres


## 2. Le vecteur : une liste qui veut dire quelque chose

### 2.1 Tu en fabriques déjà

Ayélé vend des jus frais au marché Dantokpa, à Cotonou. Sa recette
d'ananas-gingembre : 3 ananas, 2 poignées de gingembre. Une liste ordonnée de
nombres où chaque position a un sens fixé (position 0 : les ananas,
position 1 : le gingembre), c'est un **vecteur**.

In [3]:
doses = [3, 2]          # 3 ananas, 2 poignées de gingembre
print(doses)             # une simple liste Python... et déjà un vecteur

[3, 2]


En PyTorch, la liste devient un **tenseur**, l'objet avec lequel tout le livre
calcule. `.shape` (la **forme**) répond à la question « combien de nombres,
rangés comment ? » : ce petit attribut deviendra ton meilleur ami avant la fin
du chapitre.

In [4]:
doses = torch.tensor([3.0, 2.0])       # la recette
prix  = torch.tensor([250.0, 100.0])   # les prix à Dantokpa, en FCFA
print(doses.shape)                      # torch.Size([2])

torch.Size([2])


### 2.3 Les gestes de base, sous les deux regards

Un vecteur est aussi une **flèche** dans un espace : chaque dimension est un
axe. Deux gestes élémentaires : additionner deux vecteurs (position par
position, géométriquement **déplacer**) et multiplier par un nombre (chaque
position, géométriquement **étirer**).

In [5]:
lundi = torch.tensor([3.0, 2.0])    # les achats de lundi
mardi = torch.tensor([1.0, 4.0])    # ceux de mardi

print(lundi + mardi)   # tensor([4., 6.])  : le total des deux jours
print(lundi * 10)      # tensor([30., 20.]) : dix bidons au lieu d'un

tensor([4., 6.])
tensor([30., 20.])


## 3. Le produit scalaire : doser puis additionner

Combien coûte un bidon d'ananas-gingembre ? Le calcul du marché :
3 ananas × 250 FCFA + 2 poignées × 100 FCFA = 950 FCFA. Multiplier les deux
listes **position par position**, puis **tout additionner** : c'est le
**produit scalaire**. Le voici en boucle Python, exactement le geste que tu
ferais de tête au marché.

In [6]:
def produit_scalaire(u, v):
    assert u.shape == v.shape, "deux vecteurs de même longueur"
    total = 0.0
    for i in range(len(u)):              # pour chaque position...
        total += float(u[i]) * float(v[i])   # ...multiplier, puis accumuler
    return total


print(produit_scalaire(doses, prix))    # 950.0 : le coût du bidon

950.0


### 3.2 La notation, et l'outil PyTorch

En notation : $\mathbf{d} \cdot \mathbf{p} = \sum_i d_i\, p_i$, « multiplie
$d_i$ par $p_i$ pour chaque position $i$, et additionne tout ». `torch.dot`
fait tout d'un coup. Piège classique : `*` seul multiplie élément par élément
et s'arrête là ; le produit scalaire, c'est `*` **suivi de** `.sum()`.

In [7]:
print(torch.dot(doses, prix))    # tensor(950.) : le coût du bidon

# attention : * seul ne fait QUE la première moitié du geste
print(doses * prix)              # tensor([750., 200.]) : multiplié, pas encore sommé
print((doses * prix).sum())      # tensor(950.) : multiplié PUIS sommé = produit scalaire

tensor(950.)
tensor([750., 200.])
tensor(950.)


### 3.3 Le regard géométrique : une mesure de ressemblance

Le produit scalaire mesure à quel point deux flèches pointent dans la même
direction : même sens, grand et positif ; perpendiculaires, nul ;
sens opposés, négatif. Un détecteur de **ressemblance** entre vecteurs : c'est
la brique exacte du mécanisme d'attention du chapitre 9. Vérifie-le sur des
cas purs.

In [8]:
droite = torch.tensor([1.0, 0.0])
haut   = torch.tensor([0.0, 1.0])
gauche = torch.tensor([-1.0, 0.0])

print(torch.dot(droite, droite))   # tensor(1.)  : parfaitement alignés
print(torch.dot(droite, haut))     # tensor(0.)  : perpendiculaires, aucun rapport
print(torch.dot(droite, gauche))   # tensor(-1.) : opposés

tensor(1.)
tensor(0.)
tensor(-1.)


## 4. La matrice : des recettes rangées en tableau

Deux recettes (l'Ananas-gingembre : 3 et 2 ; le Gingembre corsé : 1 et 4),
deux fournisseurs (Dantokpa : ananas 250, gingembre 100 ; le supermarché :
ananas 400, gingembre 150). Un tableau à lignes et colonnes, c'est une
**matrice**. Combien coûte chaque recette chez chaque fournisseur ?

La règle du **produit matriciel** tient en une phrase : **la case (ligne i,
colonne j) du résultat est le produit scalaire de la ligne i de la première
matrice par la colonne j de la seconde.** Les trois boucles ci-dessous suivent
la règle mot pour mot ; la dernière ligne montre l'outil de tous les jours,
l'opérateur `@`.

In [9]:
D = torch.tensor([[3.0, 2.0],
                  [1.0, 4.0]])       # doses : 2 recettes x 2 ingrédients
P = torch.tensor([[250.0, 400.0],
                  [100.0, 150.0]])   # prix : 2 ingrédients x 2 fournisseurs


def matmul_a_la_main(A, B):
    m, n = A.shape                    # A : m lignes, n colonnes
    n2, p = B.shape                   # B : n2 lignes, p colonnes
    assert n == n2, "les dimensions du milieu doivent se rencontrer"
    C = torch.zeros(m, p)             # le résultat, prêt à remplir
    for i in range(m):                # pour chaque ligne de A...
        for j in range(p):            # ...et chaque colonne de B...
            for k in range(n):        # ...un produit scalaire complet
                C[i, j] += A[i, k] * B[k, j]
    return C


print(matmul_a_la_main(D, P))         # [[ 950., 1500.], [ 650., 1000.]]
print(D @ P)                          # les mêmes quatre coûts, en une ligne

tensor([[ 950., 1500.],
        [ 650., 1000.]])
tensor([[ 950., 1500.],
        [ 650., 1000.]])


### 4.3 La règle d'or des shapes, et l'erreur provoquée exprès

Pour multiplier `(m, n)` par `(n, p)`, les dimensions **du milieu** doivent se
rencontrer ; le résultat a la shape `(m, p)`, les dimensions extérieures.
Violons la règle dans un cadre calme, pour reconnaître l'erreur le jour où
elle surgira dans un vrai entraînement.

In [10]:
prix_bizarre = torch.tensor([[250.0, 400.0, 350.0]])   # shape (1, 3)

try:
    D @ prix_bizarre                    # (2, 2) @ (1, 3) : le milieu ne se rencontre pas
except RuntimeError as e:
    message = str(e)
    print("RuntimeError:", message)

assert "cannot be multiplied" in message, "PyTorch doit refuser ce produit"
print("Erreur provoquée et comprise : les dimensions du milieu, 2 et 1, ne se rencontrent pas.")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x2 and 1x3)
Erreur provoquée et comprise : les dimensions du milieu, 2 et 1, ne se rencontrent pas.


## 5. Le second regard sur la matrice : une machine à transformer

Une matrice est un tableau de nombres (une donnée) ET une transformation :
appliquée à un vecteur par produit matriciel, elle le déforme (étire, tourne,
penche). L'image à retenir : la flèche `[1, 0]` atterrit sur la **première
colonne** de la matrice, la flèche `[0, 1]` sur la **seconde**. Une matrice se
lit colonne par colonne : chaque colonne dit où part un axe.

In [11]:
M = torch.tensor([[1.5, 0.5],
                  [0.4, 1.3]])
droite = torch.tensor([1.0, 0.0])    # la flèche « un pas vers la droite »
haut   = torch.tensor([0.0, 1.0])    # la flèche « un pas vers le haut »

print(M @ droite)   # tensor([1.5000, 0.4000]) : la 1re colonne de M
print(M @ haut)     # tensor([0.5000, 1.3000]) : la 2e colonne de M

assert torch.allclose(M @ droite, M[:, 0]), "[1, 0] atterrit sur la 1re colonne"
assert torch.allclose(M @ haut,   M[:, 1]), "[0, 1] atterrit sur la 2e colonne"

tensor([1.5000, 0.4000])
tensor([0.5000, 1.3000])


## 6. Relire le fichier de MiniLM, shape par shape

Rejoue le passage d'un lot dans MiniLM, uniquement avec des shapes. Un batch
de 64 contextes de 16 caractères, la table qui donne 24 nombres par caractère,
l'aplatissement (16 × 24 = 384), puis les deux couches. `W1.T` transpose la
matrice de poids : PyTorch les range en `(sorties, entrées)`, la transposition
les remet dans le sens du produit matriciel. Et `+ b1` fonctionne grâce au
**broadcasting** : le vecteur de biais est étiré sur les 64 lignes du lot.

In [12]:
a = torch.randn(64, 384)          # le lot aplati : 64 exemples, 384 nombres chacun

W1 = etat["reseau.0.weight"]      # (192, 384)
b1 = etat["reseau.0.bias"]        # (192,)

z = a @ W1.T + b1                 # a : (64, 384), le lot aplati
print(z.shape)                    # torch.Size([64, 192])
assert z.shape == (64, 192), "règle d'or : (64, 384) @ (384, 192) donne (64, 192)"

W2 = etat["reseau.2.weight"]      # (81, 192)
b2 = etat["reseau.2.bias"]        # (81,)

scores = torch.tanh(z) @ W2.T + b2
print(scores.shape)               # torch.Size([64, 81]) : 81 scores par exemple
assert scores.shape == (64, 81), "81 scores, un par caractère possible pour la suite"

# Compte des multiplications d'un aller : le geste répété des millions de fois.
couche_1 = 64 * 192 * 384         # un produit scalaire de longueur 384 par case
couche_2 = 64 * 81 * 192
assert couche_1 == 4_718_592 and couche_2 == 995_328
print(f"multiplications par lot : {couche_1 + couche_2:,} (environ 5,7 millions)")

torch.Size([64, 192])
torch.Size([64, 81])
multiplications par lot : 5,713,920 (environ 5,7 millions)


### Le piège du broadcasting silencieux

Le broadcasting rend service, mais il ne devine pas ton intention : si tes
shapes ne sont pas celles que tu crois, il peut étirer quand même et produire
un résultat faux **sans aucune erreur**. Le cas d'école : une colonne `(3, 1)`
plus une ligne `(1, 3)`.

In [13]:
colonne = torch.tensor([[1.0], [2.0], [3.0]])   # (3, 1)
ligne   = torch.tensor([[10.0, 20.0, 30.0]])    # (1, 3)
print((colonne + ligne).shape)                   # torch.Size([3, 3]) !

assert (colonne + ligne).shape == (3, 3), (
    "le bug qui ne crie pas : chaque dimension de taille 1 est étirée face au 3 d'en face"
)
print("Tu attendais 3 nombres, tu obtiens une matrice 3 x 3, et rien ne plante.")
print("L'antidote : le réflexe shape, après chaque opération.")

torch.Size([3, 3])
Tu attendais 3 nombres, tu obtiens une matrice 3 x 3, et rien ne plante.
L'antidote : le réflexe shape, après chaque opération.


## Exercices

À toi de jouer : trois exercices, du plus simple (●) au plus costaud (●●●).
Chaque cellule marquée `# TODO(toi)` contient un trou ; complète-le, puis
exécute la cellule de validation (`assert`) qui suit : si elle passe sans
erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Produit scalaire et matmul, version PyTorch — niveau ●

La version à trois boucles de la leçon, tu ne l'écriras plus jamais : PyTorch
fait la même chose en beaucoup plus rapide. `torch.dot` pour le produit
scalaire, l'opérateur `@` pour le produit matriciel. Refais les deux calculs
d'Ayélé avec les outils de tous les jours.

In [14]:
# TODO(toi) : refais les deux calculs avec PyTorch.
# 1) cout_bidon : le produit scalaire doses . prix, avec torch.dot(...)
# 2) couts_torch : le produit matriciel D par P, avec l'opérateur @
cout_bidon = torch.dot(doses, prix)
couts_torch = D @ P

print(cout_bidon)      # attendu : tensor(950.)
print(couts_torch)     # attendu : [[ 950., 1500.], [ 650., 1000.]]

tensor(950.)
tensor([[ 950., 1500.],
        [ 650., 1000.]])


In [15]:
# Validation : la main et la machine doivent dire exactement la même chose.
attendu = torch.tensor([[950.0, 1500.0],
                        [650.0, 1000.0]])
assert float(cout_bidon) == 950.0, "torch.dot(doses, prix) doit valoir 950"
assert torch.allclose(couts_torch, attendu), "D @ P doit donner les quatre coûts"

# Au passage : * seul ne fait QUE la première moitié du geste.
assert torch.allclose(doses * prix, torch.tensor([750.0, 200.0]))
assert float((doses * prix).sum()) == 950.0    # multiplié PUIS sommé = produit scalaire

print("Vérification OK : @ fabrique bien une collection de produits scalaires.")

Vérification OK : @ fabrique bien une collection de produits scalaires.


### Exercice 2 · Le produit scalaire à la main — niveau ●●

Réécris `produit_scalaire`, sans regarder la leçon : multiplier les deux
vecteurs **position par position**, puis **tout additionner**, avec une simple
boucle Python. Interdit ici : `torch.dot`, `@`, `(u * v).sum()`. C'est toi le
processeur.

In [16]:
def produit_scalaire(u, v):
    assert u.shape == v.shape, "deux vecteurs de même longueur"
    total = 0.0
    for i in range(u.shape[0]): # TODO(toi) : pour chaque position i, multiplie u[i] par v[i]
        total += u[i] * v[i] # et accumule le résultat dans total.
    # Indice : for i in range(len(u)): ...
    return total


print(produit_scalaire(doses, prix))    # attendu : 950.0

tensor(950.)


In [17]:
# Validation du produit scalaire : le coût du bidon, puis les cas purs.
assert abs(produit_scalaire(doses, prix) - 950.0) < 1e-6, (
    "doses . prix doit valoir 950 FCFA (3 x 250 + 2 x 100)"
)

nouvelle = torch.tensor([2.0, 3.0])      # 2 ananas, 3 poignées de gingembre
assert abs(produit_scalaire(nouvelle, prix) - 800.0) < 1e-6, (
    "2 x 250 + 3 x 100 = 800 FCFA"
)

droite = torch.tensor([1.0, 0.0])
haut   = torch.tensor([0.0, 1.0])
gauche = torch.tensor([-1.0, 0.0])
assert produit_scalaire(droite, droite) == 1.0,  "alignées : grand et positif"
assert produit_scalaire(droite, haut)   == 0.0,  "perpendiculaires : zéro"
assert produit_scalaire(droite, gauche) == -1.0, "opposées : négatif"

print("Produit scalaire OK : le bidon coûte 950 FCFA, et les flèches parlent.")

Produit scalaire OK : le bidon coûte 950 FCFA, et les flèches parlent.


### Exercice 3 · Le produit matriciel à la main — niveau ●●●

La règle tient en une phrase : **la case (ligne i, colonne j) du résultat est
le produit scalaire de la ligne i de la première matrice par la colonne j de
la seconde.** Écris les trois boucles, sans regarder la leçon : c'est le calcul
le plus important du livre, et tes doigts doivent connaître le geste.

In [18]:
def matmul_a_la_main(A, B):
    m, n = A.shape                    # A : m lignes, n colonnes
    n2, p = B.shape                   # B : n2 lignes, p colonnes
    assert n == n2, "les dimensions du milieu doivent se rencontrer"
    C = torch.zeros(m, p)             # le résultat, prêt à remplir
    # TODO(toi) : trois boucles imbriquées.
    # Pour chaque ligne i de A (range(m)), pour chaque colonne j de B (range(p)),
    # accumule dans C[i, j] les produits A[i, k] * B[k, j] pour k dans range(n).
    # La boucle sur k, c'est un produit scalaire complet.
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C


print(matmul_a_la_main(D, P))         # attendu : [[ 950., 1500.], [ 650., 1000.]]

tensor([[ 950., 1500.],
        [ 650., 1000.]])


In [19]:
# Validation du produit matriciel : les quatre coûts d'Ayélé.
attendu = torch.tensor([[950.0, 1500.0],
                        [650.0, 1000.0]])
couts = matmul_a_la_main(D, P)
assert couts.shape == (2, 2), f"shape attendue (2, 2), obtenue {tuple(couts.shape)}"
assert torch.allclose(couts, attendu), (
    "chaque case (i, j) = produit scalaire de la ligne i de D par la colonne j de P"
)
assert torch.allclose(couts, D @ P), "@ et tes trois boucles font le même calcul"
print("Produit matriciel OK : 950, 1500, 650 et 1000 FCFA, comme au marché.")

Produit matriciel OK : 950, 1500, 650 et 1000 FCFA, comme au marché.


## Verdict

Trois validations vertes : le fichier de MiniLM n'est plus une boîte opaque.
Cinq paquets, trois matrices, deux vecteurs, et un geste qui les anime : le
produit matriciel, que tes doigts ont fait à la main sur les recettes d'Ayélé.
Ce qui manque encore : comment on **trouve** les bonnes valeurs de ces
91 497 nombres.

Retour au livre pour la suite : *Chapitre 3 · Comment une machine apprend*.